# 04 ? Retrieval Benchmark: RepoBench Baseline Comparison
Build two retrieval tables on RepoBench v1.1 Python `cross_file_first`:
- strict paper-style table: last-3-lines query, only examples with `>=5` candidates, easy/hard buckets
- enhanced table: full `cropped_code` query plus stronger embeddings
When `test_indices.pt` exists, this notebook evaluates only that held-out subset. Per-example rankings and final tables are saved to `data/results/`.

## Setup

In [1]:
!pip install torch transformers datasets rank-bm25 tqdm sentencepiece --quiet

In [2]:
import sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/HaluGuard'
except ImportError:
    REPO_DIR = os.path.abspath('..')
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
!pip install -e '.[dev]' -q
from notebooks.utils import check_gpu, get_drive_path
DEVICE = check_gpu()

Mounted at /content/drive
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.0 MB/s eta 0:00:00
  Building editable for haluguard (pyproject.toml) ... done
GPU found: Tesla T4 (15.6 GB VRAM)


In [3]:
import json, warnings
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple
import torch
from tqdm import tqdm
from datasets import load_dataset
from haluguard.baselines import cosine_scores, edit_similarity_scores, jaccard_scores, random_ranking
from haluguard.hccs import HCCSScorer
from haluguard.retrieval_benchmark import QUERY_VIEW_FULL, QUERY_VIEW_LAST3, build_query_text, build_ranking_result, get_candidate_bucket, get_chunk_embedding_path, get_hccs_checkpoint_name, get_query_embedding_path, rank_indices_from_scores, save_rankings_jsonl, save_table_json, summarise_rankings
DATA_DIR = get_drive_path('HaluGuard/data')
EMB_DIR = DATA_DIR / 'embeddings'
CKPT_DIR = get_drive_path('HaluGuard/checkpoints')
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_VARIANTS = [
    {'backend': 'codebert', 'query_view': QUERY_VIEW_LAST3, 'display_name': 'CodeBERT'},
    {'backend': 'codebert', 'query_view': QUERY_VIEW_FULL, 'display_name': 'CodeBERT'},
    {'backend': 'unixcoder', 'query_view': QUERY_VIEW_LAST3, 'display_name': 'UniXcoder'},
    {'backend': 'unixcoder', 'query_view': QUERY_VIEW_FULL, 'display_name': 'UniXcoder'},
]
STRICT_VARIANTS = [variant for variant in BENCHMARK_VARIANTS if variant['query_view'] == QUERY_VIEW_LAST3]
STRICT_METHOD_ORDER = ['Random', 'Jaccard', 'Edit', 'CodeBERT cosine last3', 'UniXcoder cosine last3', 'HCCS-CodeBERT-last3', 'HCCS-UniXcoder-last3']
ENHANCED_METHOD_ORDER = ['CodeBERT cosine last3', 'CodeBERT cosine full', 'UniXcoder cosine last3', 'UniXcoder cosine full', 'HCCS-CodeBERT-last3', 'HCCS-CodeBERT-full', 'HCCS-UniXcoder-last3', 'HCCS-UniXcoder-full']
_QUERY_EMBED_CACHE: Dict[Tuple[str, str], Tuple[torch.Tensor, Path, Dict[str, Any]]] = {}
_CHUNK_EMBED_CACHE: Dict[str, Tuple[Sequence[torch.Tensor], Path]] = {}
_SCORER_CACHE: Dict[Tuple[str, str], HCCSScorer] = {}


def _load_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def _variant_method_name(prefix: str, display_name: str, query_view: str) -> str:
    if prefix == 'cosine':
        return f'{display_name} cosine {query_view}'
    return f'HCCS-{display_name}-{query_view}'


def _load_query_embeddings(backend: str, query_view: str):
    cache_key = (backend, query_view)
    if cache_key in _QUERY_EMBED_CACHE:
        return _QUERY_EMBED_CACHE[cache_key]

    query_path = get_query_embedding_path(EMB_DIR, backend, query_view)
    query_meta_path = query_path.with_suffix('.meta.json')
    if query_path.exists():
        payload = (torch.load(query_path), query_path, _load_json(query_meta_path))
        _QUERY_EMBED_CACHE[cache_key] = payload
        return payload

    legacy_path = EMB_DIR / 'query_embeddings.pt'
    legacy_meta_path = EMB_DIR / 'query_embeddings_meta.json'
    if backend == 'codebert' and query_view == QUERY_VIEW_FULL and legacy_path.exists():
        payload = (torch.load(legacy_path), legacy_path, _load_json(legacy_meta_path))
        _QUERY_EMBED_CACHE[cache_key] = payload
        return payload

    raise FileNotFoundError(f'Missing query embeddings for {backend}/{query_view}. Run notebook 01 first.')


def _load_chunk_embeddings(backend: str):
    if backend in _CHUNK_EMBED_CACHE:
        return _CHUNK_EMBED_CACHE[backend]

    chunk_path = get_chunk_embedding_path(EMB_DIR, backend)
    if chunk_path.exists():
        payload = (torch.load(chunk_path), chunk_path)
        _CHUNK_EMBED_CACHE[backend] = payload
        return payload

    legacy_path = EMB_DIR / 'chunk_embeddings.pt'
    if backend == 'codebert' and legacy_path.exists():
        payload = (torch.load(legacy_path), legacy_path)
        _CHUNK_EMBED_CACHE[backend] = payload
        return payload

    raise FileNotFoundError(f'Missing chunk embeddings for {backend}. Run notebook 01 first.')


def _load_hccs_scorer(backend: str, query_view: str) -> HCCSScorer:
    cache_key = (backend, query_view)
    if cache_key in _SCORER_CACHE:
        return _SCORER_CACHE[cache_key]

    checkpoint_path = CKPT_DIR / get_hccs_checkpoint_name(backend, query_view)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint {checkpoint_path.name}. Train it in notebook 02 first.')
    scorer = HCCSScorer.load(checkpoint_path)
    scorer = scorer.to(DEVICE)
    scorer.eval()
    _SCORER_CACHE[cache_key] = scorer
    return scorer


def _evaluate_ranked_method(method_name: str, examples: Sequence[Dict[str, Any]], query_view: str, rank_fn) -> List[Any]:
    rankings = []
    for example in tqdm(examples, desc=method_name):
        ranked_indices = rank_fn(example)
        rankings.append(build_ranking_result(method=method_name, example_id=example['example_id'], query_view=query_view, candidate_count=example['candidate_count'], gold_index=example['gold_index'], ranked_indices=ranked_indices))
    return rankings


def _evaluate_cosine_method(method_name: str, examples: Sequence[Dict[str, Any]], query_view: str, query_embs: torch.Tensor, chunk_embs: Sequence[torch.Tensor]) -> List[Any]:
    return _evaluate_ranked_method(method_name, examples, query_view, lambda example: rank_indices_from_scores(cosine_scores(query_embs[example['example_id']].numpy(), chunk_embs[example['example_id']].numpy())))


def _evaluate_hccs_method(method_name: str, examples: Sequence[Dict[str, Any]], query_view: str, query_embs: torch.Tensor, chunk_embs: Sequence[torch.Tensor], scorer: HCCSScorer) -> List[Any]:
    return _evaluate_ranked_method(method_name, examples, query_view, lambda example: rank_indices_from_scores(scorer.score_chunks(query_embs[example['example_id']].numpy(), chunk_embs[example['example_id']].numpy(), device=DEVICE)))


def _order_table(table: List[Dict[str, Any]], method_order: Sequence[str]) -> List[Dict[str, Any]]:
    row_map = {row['method']: row for row in table}
    ordered = [row_map[name] for name in method_order if name in row_map]
    remaining = [row for row in table if row['method'] not in set(method_order)]
    return ordered + sorted(remaining, key=lambda row: row['method'])


def _fmt_pct(row: Dict[str, Any], key: str) -> str:
    return f"{row[key]:.1%}" if key in row else '-'


def _print_table(title: str, table: List[Dict[str, Any]]) -> None:
    print()
    print(title)
    print(f"{'Method':<24} {'easy_n':>7} {'easy@1':>8} {'easy@3':>8} {'hard_n':>7} {'hard@1':>8} {'hard@3':>8} {'hard@5':>8}")
    print('-' * 86)
    for row in table:
        print(f"{row['method']:<24} {row.get('easy_count', 0):>7d} {_fmt_pct(row, 'easy_acc@1'):>8} {_fmt_pct(row, 'easy_acc@3'):>8} {row.get('hard_count', 0):>7d} {_fmt_pct(row, 'hard_acc@1'):>8} {_fmt_pct(row, 'hard_acc@3'):>8} {_fmt_pct(row, 'hard_acc@5'):>8}")


print('Benchmark variants:')
for variant in BENCHMARK_VARIANTS:
    print(f"  - {variant['display_name']} / {variant['query_view']}")


Benchmark variants:
  - CodeBERT / last3
  - CodeBERT / full
  - UniXcoder / last3
  - UniXcoder / full


## Evaluation Subset

In [4]:
ds = load_dataset('tianyang/repobench_python_v1.1', split='cross_file_first')
test_indices_path = EMB_DIR / 'test_indices.pt'
if test_indices_path.exists():
    allowed_ids = sorted(int(index) for index in torch.load(test_indices_path))
    print(f'Using held-out test indices from {test_indices_path.name}: {len(allowed_ids)} examples before filtering')
else:
    warnings.warn('test_indices.pt not found. Falling back to all eligible RepoBench examples. Run notebook 02 first for a held-out evaluation subset.')
    allowed_ids = list(range(len(ds)))
benchmark_examples: List[Dict[str, Any]] = []
for example_id in allowed_ids:
    example = ds[example_id]
    candidate_count = len(example['context'])
    gold_index = int(example['gold_snippet_index'])
    bucket = get_candidate_bucket(candidate_count)
    if bucket is None or gold_index < 0 or gold_index >= candidate_count:
        continue
    benchmark_examples.append({'example_id': example_id, 'cropped_code': example['cropped_code'], 'contexts': example['context'], 'candidate_count': candidate_count, 'gold_index': gold_index, 'bucket': bucket})
print(f'Benchmark examples after filtering: {len(benchmark_examples)}')
print(f"Easy (5-9 candidates): {sum(example['bucket'] == 'easy' for example in benchmark_examples)}")
print(f"Hard (10+ candidates): {sum(example['bucket'] == 'hard' for example in benchmark_examples)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/cross_file_first-00000-of-00002-bae(…):   0%|          | 0.00/32.1M [00:00<?, ?B/s]

data/cross_file_first-00001-of-00002-578(…):   0%|          | 0.00/132M [00:00<?, ?B/s]

data/cross_file_random-00000-of-00001-52(…):   0%|          | 0.00/153M [00:00<?, ?B/s]

data/in_file-00000-of-00001-9721c68f6224(…):   0%|          | 0.00/156M [00:00<?, ?B/s]

Generating cross_file_first split:   0%|          | 0/8033 [00:00<?, ? examples/s]

Generating cross_file_random split:   0%|          | 0/7618 [00:00<?, ? examples/s]

Generating in_file split:   0%|          | 0/7910 [00:00<?, ? examples/s]

Using held-out test indices from test_indices.pt: 804 examples before filtering
Benchmark examples after filtering: 591
Easy (5-9 candidates): 252
Hard (10+ candidates): 339


## Strict Paper-Style Table

In [5]:
strict_rankings = []
strict_rankings.extend(_evaluate_ranked_method('Random', benchmark_examples, QUERY_VIEW_LAST3, lambda example: random_ranking(example['candidate_count'], seed=42 + int(example['example_id']))))
strict_rankings.extend(_evaluate_ranked_method('Jaccard', benchmark_examples, QUERY_VIEW_LAST3, lambda example: rank_indices_from_scores(jaccard_scores(build_query_text(example['cropped_code'], QUERY_VIEW_LAST3), example['contexts']))))
strict_rankings.extend(_evaluate_ranked_method('Edit', benchmark_examples, QUERY_VIEW_LAST3, lambda example: rank_indices_from_scores(edit_similarity_scores(build_query_text(example['cropped_code'], QUERY_VIEW_LAST3), example['contexts']))))

for variant in STRICT_VARIANTS:
    backend = variant['backend']
    query_view = variant['query_view']
    display_name = variant['display_name']

    try:
        query_embs, query_path, _ = _load_query_embeddings(backend, query_view)
        chunk_embs, chunk_path = _load_chunk_embeddings(backend)
    except FileNotFoundError as error:
        warnings.warn(str(error))
        continue

    print(f'{display_name} strict query embeddings: {query_path.name}')
    print(f'{display_name} strict chunk embeddings: {chunk_path.name}')
    strict_rankings.extend(_evaluate_cosine_method(_variant_method_name('cosine', display_name, query_view), benchmark_examples, query_view, query_embs, chunk_embs))

    try:
        scorer = _load_hccs_scorer(backend, query_view)
        strict_rankings.extend(_evaluate_hccs_method(_variant_method_name('hccs', display_name, query_view), benchmark_examples, query_view, query_embs, chunk_embs, scorer))
    except FileNotFoundError as error:
        warnings.warn(str(error))

strict_table = _order_table(summarise_rankings(strict_rankings), STRICT_METHOD_ORDER)
strict_rankings_path = RESULTS_DIR / 'retrieval_benchmark_strict_rankings.jsonl'
strict_table_path = RESULTS_DIR / 'retrieval_benchmark_strict_table.json'
save_rankings_jsonl(strict_rankings, strict_rankings_path)
save_table_json(strict_table, strict_table_path)
print(f'Saved strict rankings -> {strict_rankings_path}')
print(f'Saved strict table -> {strict_table_path}')
_print_table('Strict paper-style retrieval table', strict_table)


Edit: 100%|██████████| 591/591 [00:03<00:00, 159.40it/s]


CodeBERT strict query embeddings: query_embeddings__codebert__last3.pt
CodeBERT strict chunk embeddings: chunk_embeddings__codebert.pt


HCCS-CodeBERT-last3: 100%|██████████| 591/591 [00:00<00:00, 805.02it/s]


UniXcoder strict query embeddings: query_embeddings__unixcoder__last3.pt
UniXcoder strict chunk embeddings: chunk_embeddings__unixcoder.pt


HCCS-UniXcoder-last3: 100%|██████████| 591/591 [00:00<00:00, 1874.32it/s]


Saved strict rankings -> /content/drive/MyDrive/HaluGuard/data/results/retrieval_benchmark_strict_rankings.jsonl
Saved strict table -> /content/drive/MyDrive/HaluGuard/data/results/retrieval_benchmark_strict_table.json

Strict paper-style retrieval table
Method                    easy_n   easy@1   easy@3  hard_n   hard@1   hard@3   hard@5
--------------------------------------------------------------------------------------
Random                       252    18.3%    43.3%     339     5.9%    17.4%    31.0%
Jaccard                      252    15.9%    50.0%     339     6.5%    18.6%    31.0%
Edit                         252    16.3%    49.6%     339     8.6%    20.4%    33.3%
CodeBERT cosine last3        252    19.0%    51.2%     339     8.0%    19.5%    32.7%
UniXcoder cosine last3       252    28.2%    57.5%     339    18.3%    42.8%    58.4%
HCCS-CodeBERT-last3          252    15.9%    52.4%     339     9.1%    20.6%    35.7%
HCCS-UniXcoder-last3         252    28.2%    60.7%     3

## Enhanced Table

In [6]:
enhanced_rankings = []

for variant in BENCHMARK_VARIANTS:
    backend = variant['backend']
    query_view = variant['query_view']
    display_name = variant['display_name']

    try:
        query_embs, query_path, _ = _load_query_embeddings(backend, query_view)
        chunk_embs, chunk_path = _load_chunk_embeddings(backend)
    except FileNotFoundError as error:
        warnings.warn(str(error))
        continue

    print(f'{display_name} {query_view} query embeddings: {query_path.name}')
    print(f'{display_name} {query_view} chunk embeddings: {chunk_path.name}')
    enhanced_rankings.extend(_evaluate_cosine_method(_variant_method_name('cosine', display_name, query_view), benchmark_examples, query_view, query_embs, chunk_embs))

    try:
        scorer = _load_hccs_scorer(backend, query_view)
        enhanced_rankings.extend(_evaluate_hccs_method(_variant_method_name('hccs', display_name, query_view), benchmark_examples, query_view, query_embs, chunk_embs, scorer))
    except FileNotFoundError as error:
        warnings.warn(str(error))

enhanced_table = _order_table(summarise_rankings(enhanced_rankings), ENHANCED_METHOD_ORDER)
enhanced_rankings_path = RESULTS_DIR / 'retrieval_benchmark_enhanced_rankings.jsonl'
enhanced_table_path = RESULTS_DIR / 'retrieval_benchmark_enhanced_table.json'
save_rankings_jsonl(enhanced_rankings, enhanced_rankings_path)
save_table_json(enhanced_table, enhanced_table_path)
print(f'Saved enhanced rankings -> {enhanced_rankings_path}')
print(f'Saved enhanced table -> {enhanced_table_path}')
_print_table('Enhanced retrieval table', enhanced_table)


CodeBERT last3 query embeddings: query_embeddings__codebert__last3.pt
CodeBERT last3 chunk embeddings: chunk_embeddings__codebert.pt


HCCS-CodeBERT-last3: 100%|██████████| 591/591 [00:00<00:00, 1837.81it/s]


CodeBERT full query embeddings: query_embeddings__codebert__full.pt
CodeBERT full chunk embeddings: chunk_embeddings__codebert.pt


HCCS-CodeBERT-full: 100%|██████████| 591/591 [00:00<00:00, 1912.22it/s]


UniXcoder last3 query embeddings: query_embeddings__unixcoder__last3.pt
UniXcoder last3 chunk embeddings: chunk_embeddings__unixcoder.pt


HCCS-UniXcoder-last3: 100%|██████████| 591/591 [00:00<00:00, 1920.48it/s]

Saved enhanced rankings -> /content/drive/MyDrive/HaluGuard/data/results/retrieval_benchmark_enhanced_rankings.jsonl
Saved enhanced table -> /content/drive/MyDrive/HaluGuard/data/results/retrieval_benchmark_enhanced_table.json

Enhanced retrieval table
Method                    easy_n   easy@1   easy@3  hard_n   hard@1   hard@3   hard@5
--------------------------------------------------------------------------------------
CodeBERT cosine last3        252    19.0%    51.2%     339     8.0%    19.5%    32.7%
CodeBERT cosine full         252    13.9%    43.3%     339     8.8%    23.9%    36.0%
UniXcoder cosine last3       252    28.2%    57.5%     339    18.3%    42.8%    58.4%
HCCS-CodeBERT-last3          252    15.9%    52.4%     339     9.1%    20.6%    35.7%
HCCS-CodeBERT-full           252    17.9%    53.2%     339     7.7%    21.5%    32.4%
HCCS-UniXcoder-last3         252    28.2%    60.7%     339    17.4%    41.3%    53.4%



/tmp/ipykernel_1125/4276917229.py:12: UserWarning: Missing query embeddings for unixcoder/full. Run notebook 01 first.
  warnings.warn(str(error))
